In [1]:
# ============================================================
# CELL 1: Install Dependencies
# ============================================================
!pip -q install torchcde pyyaml


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.2/61.2 kB 3.5 MB/s eta 0:00:00


In [2]:
# ============================================================
# CELL 2: Setup Environment
# ============================================================
import os, json, pickle
import numpy as np
import pandas as pd

os.chdir("/kaggle/working")
print("✅ Environment ready")

✅ Environment ready


In [3]:
# ============================================================
# CELL 3: Load All METR-LA Files
# ============================================================

# Adapt this path to your Kaggle dataset
INPUT_DIR = "/kaggle/input/datasets/cheminichemseddine/metr-la-complete"  # Change if different

# Load dataset
df = pd.read_hdf(f"{INPUT_DIR}/metr_la.h5")
print(f"✅ metr_la.h5: {df.shape} ({df.shape[0]} timesteps × {df.shape[1]} sensors)")

# Load mean/std
with open(f"{INPUT_DIR}/metr_meanstd.pk", 'rb') as f:
    train_mean, train_std = pickle.load(f)
print(f"✅ metr_meanstd.pk: mean={train_mean.shape}, std={train_std.shape}")

# Load distance matrix
dist_matrix = np.load(f"{INPUT_DIR}/metr_la_dist.npy")
print(f"✅ metr_la_dist.npy: {dist_matrix.shape}")

# Load sensor locations
df_locations = pd.read_csv(f"{INPUT_DIR}/sensor_locations_la.csv")
print(f"✅ sensor_locations_la.csv: {df_locations.shape}")

# Load sensor IDs
with open(f"{INPUT_DIR}/sensor_ids_la.txt", 'r') as f:
    sensor_ids = [s.strip() for s in f.read().strip().split(',')]
print(f"✅ sensor_ids_la: {len(sensor_ids)} sensors")

# Verify consistency
assert len(sensor_ids) == df.shape[1], "Sensor count mismatch!"
assert dist_matrix.shape == (len(sensor_ids), len(sensor_ids)), "Distance matrix shape mismatch!"
assert len(df_locations) == len(sensor_ids), "Location count mismatch!"

print(f"\n{'='*60}")
print(f"ALL FILES CONSISTENT: {len(sensor_ids)} sensors")
print(f"{'='*60}")



✅ metr_la.h5: (34272, 207) (34272 timesteps × 207 sensors)
✅ metr_meanstd.pk: mean=(207,), std=(207,)
✅ metr_la_dist.npy: (207, 207)
✅ sensor_locations_la.csv: (207, 4)
✅ sensor_ids_la: 207 sensors

ALL FILES CONSISTENT: 207 sensors


In [4]:
# ============================================================
# CELL 4: Explore the Data
# ============================================================
print("=" * 60)
print("DATA EXPLORATION")
print("=" * 60)

# Time range
print(f"\nTime range: {df.index[0]} → {df.index[-1]}")
print(f"Granularity: 5-minute intervals")
print(f"Total timesteps: {len(df)}")

# Data quality
X = df.values
zero_mask = (X == 0.)
print(f"\nData quality:")
print(f"  Zero values (missing): {zero_mask.sum()} ({100*zero_mask.mean():.1f}%)")
print(f"  Value range: [{X[X>0].min():.2f}, {X.max():.2f}]")
print(f"  Mean (non-zero): {X[X>0].mean():.2f}")

# Per-sensor missing rate
missing_per_sensor = zero_mask.mean(axis=0)
print(f"\nPer-sensor missing rate:")
print(f"  Min: {missing_per_sensor.min()*100:.1f}%")
print(f"  Max: {missing_per_sensor.max()*100:.1f}%")
print(f"  Mean: {missing_per_sensor.mean()*100:.1f}%")

# Distance matrix
finite_dist = dist_matrix[~np.isinf(dist_matrix) & (dist_matrix > 0)]
print(f"\nDistance matrix:")
print(f"  Finite distances: {len(finite_dist)}")
print(f"  Inf distances: {np.isinf(dist_matrix).sum()}")
print(f"  Range: [{finite_dist.min():.1f}, {finite_dist.max():.1f}] meters")

# Sensor locations
print(f"\nSensor locations:")
print(f"  Lat range: [{df_locations['latitude'].min():.4f}, {df_locations['latitude'].max():.4f}]")
print(f"  Lon range: [{df_locations['longitude'].min():.4f}, {df_locations['longitude'].max():.4f}]")

DATA EXPLORATION

Time range: 2012-03-01 00:00:00 → 2012-06-27 23:55:00
Granularity: 5-minute intervals
Total timesteps: 34272

Data quality:
  Zero values (missing): 575302 (8.1%)
  Value range: [0.33, 70.00]
  Mean (non-zero): 58.46

Per-sensor missing rate:
  Min: 6.3%
  Max: 20.1%
  Mean: 8.1%

Distance matrix:
  Finite distances: 11546
  Inf distances: 31096
  Range: [33.5, 11895.3] meters

Sensor locations:
  Lat range: [34.0430, 34.2216]
  Lon range: [-118.5368, -118.1829]


In [5]:
# ============================================================
# CELL 5: Generate Node Metadata
# ============================================================
print("=" * 60)
print("GENERATING NODE METADATA")
print("=" * 60)

X = df.values  # (34272, 207)
T, N = X.shape

# Use 70% for training stats (same as PriSTI)
train_end = int(T * 0.7)
X_train = X[:train_end]

nodes_metadata = []

for j in range(N):
    sensor_data = X[:, j]
    sensor_train = X_train[:, j]

    # Observed values (non-zero)
    observed = sensor_data[sensor_data != 0.]
    observed_train = sensor_train[sensor_train != 0.]

    # Missing rate
    n_missing = (sensor_data == 0.).sum()
    missing_rate = n_missing / T

    # Category
    if missing_rate < 0.1:
        missing_category = "low"
    elif missing_rate < 0.3:
        missing_category = "medium"
    else:
        missing_category = "high"

    # Statistics (on training data, non-zero)
    if len(observed_train) > 0:
        mean_val = float(np.mean(observed_train))
        std_val = float(np.std(observed_train))
        min_val = float(np.min(observed_train))
        max_val = float(np.max(observed_train))
    else:
        mean_val, std_val, min_val, max_val = 0., 1., 0., 0.

    # Temporal variance (24-step windows = 2 hours)
    window_size = 24
    variances = []
    for t in range(0, train_end - window_size, window_size):
        window = sensor_train[t:t+window_size]
        observed_in_window = window[window != 0.]
        if len(observed_in_window) > 1:
            variances.append(float(np.var(observed_in_window)))

    if len(variances) > 0:
        temporal_var_mean = float(np.mean(variances))
        temporal_var_std = float(np.std(variances))
    else:
        temporal_var_mean = 0.
        temporal_var_std = 1e10

    # Stability
    stability = "stable" if temporal_var_std < 2 * temporal_var_mean else "unstable"

    # Autocorrelation lag 1
    if len(observed_train) > 10:
        autocorr_lag1 = float(np.corrcoef(observed_train[:-1], observed_train[1:])[0, 1])
        if np.isnan(autocorr_lag1):
            autocorr_lag1 = 0.
    else:
        autocorr_lag1 = 0.

    # Smoothness
    smoothness_label = "smooth" if autocorr_lag1 > 0.8 else "rough"

    # GPS coordinates
    lat = float(df_locations.iloc[j]['latitude'])
    lon = float(df_locations.iloc[j]['longitude'])

    node = {
        "sensor_id": str(sensor_ids[j]),
        "sensor_index": j,
        "latitude": lat,
        "longitude": lon,
        "n_missing": int(n_missing),
        "n_total": T,
        "missing_rate": round(float(missing_rate), 4),
        "missing_category": missing_category,
        "mean": round(mean_val, 4),
        "std": round(std_val, 4),
        "min_val": round(min_val, 4),
        "max_val": round(max_val, 4),
        "temporal_var_mean": round(temporal_var_mean, 4),
        "temporal_var_std": round(temporal_var_std, 4),
        "stability": stability,
        "autocorr_lag1": round(autocorr_lag1, 4),
        "smoothness_label": smoothness_label
    }
    nodes_metadata.append(node)

    if j < 3 or j == N-1:
        print(f"  Sensor {j} ({sensor_ids[j]}): missing={missing_rate*100:.1f}%, "
              f"mean={mean_val:.1f}, var_mean={temporal_var_mean:.1f}")

print(f"\n✅ {len(nodes_metadata)} node metadata generated")


GENERATING NODE METADATA
  Sensor 0 (773869): missing=11.2%, mean=61.9, var_mean=51.2
  Sensor 1 (767541): missing=6.3%, mean=64.4, var_mean=6.2
  Sensor 2 (767542): missing=6.3%, mean=64.7, var_mean=39.2
  Sensor 206 (769373): missing=6.3%, mean=55.2, var_mean=78.8

✅ 207 node metadata generated


In [6]:
# ============================================================
# CELL 6: Generate Relation Metadata
# ============================================================
print("=" * 60)
print("GENERATING RELATION METADATA")
print("=" * 60)

# Compute adjacency like PriSTI does
finite_dist = dist_matrix.reshape(-1)
finite_dist = finite_dist[~np.isinf(finite_dist)]
sigma = finite_dist.std()
adj = np.exp(-np.square(dist_matrix / sigma))
adj[adj < 0.1] = 0.
np.fill_diagonal(adj, 0.)

print(f"  Sigma: {sigma:.2f}")
print(f"  Adjacency non-zero: {np.count_nonzero(adj)}")

# Compute Pearson correlations on training data
print("  Computing Pearson correlations (this may take a minute)...")
X_train_norm = X_train.copy()
ob_mask_train = (X_train != 0.).astype(float)

relations_metadata = []
n_relations = 0

for i in range(N):
    for j in range(i+1, N):
        gaussian_weight = float(adj[i, j])

        # Only create relation if gaussian_weight > 0 or sensors are interesting
        if gaussian_weight == 0.:
            continue

        # Pearson correlation on co-observed values
        mask_both = (X_train[:, i] != 0.) & (X_train[:, j] != 0.)
        n_co_obs = mask_both.sum()

        if n_co_obs > 10:
            pearson = float(np.corrcoef(X_train[mask_both, i], X_train[mask_both, j])[0, 1])
            if np.isnan(pearson):
                pearson = 0.
        else:
            pearson = 0.

        # Distance
        dist_val = float(dist_matrix[i, j])
        if np.isinf(dist_val):
            dist_km = -1.
        else:
            dist_km = round(dist_val / 1000., 2)  # meters to km

        # Relation type
        if pearson >= 0.85:
            rel_type = "STRONGLY_CORRELATED"
        elif pearson >= 0.7:
            rel_type = "CORRELATED"
        else:
            rel_type = "WEAKLY_CORRELATED"

        rel = {
            "source": str(sensor_ids[i]),
            "target": str(sensor_ids[j]),
            "pearson": round(pearson, 4),
            "weight": round(pearson, 4),
            "distance_km": dist_km,
            "gaussian_weight": round(gaussian_weight, 6),
            "rel_type": rel_type
        }
        relations_metadata.append(rel)
        n_relations += 1

print(f"\n✅ {n_relations} relations generated")

# Stats
pearson_vals = [r['pearson'] for r in relations_metadata]
gw_vals = [r['gaussian_weight'] for r in relations_metadata]
print(f"  Pearson range: [{min(pearson_vals):.4f}, {max(pearson_vals):.4f}]")
print(f"  Gaussian weight range: [{min(gw_vals):.4f}, {max(gw_vals):.4f}]")



GENERATING RELATION METADATA
  Sigma: 2584.45
  Adjacency non-zero: 1515
  Computing Pearson correlations (this may take a minute)...

✅ 776 relations generated
  Pearson range: [-0.1669, 0.9794]
  Gaussian weight range: [0.1001, 0.9995]


In [7]:
# ============================================================
# CELL 7: Save Everything
# ============================================================
OUTPUT_DIR = "/kaggle/working/metrla_metadata"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Save nodes
with open(f"{OUTPUT_DIR}/nodes_metadata.json", 'w') as f:
    json.dump(nodes_metadata, f, indent=2)
print(f"✅ nodes_metadata.json: {len(nodes_metadata)} nodes")

# Save relations (including gaussian_weight already!)
with open(f"{OUTPUT_DIR}/relations_metadata.json", 'w') as f:
    json.dump(relations_metadata, f, indent=2)
print(f"✅ relations_metadata.json: {len(relations_metadata)} relations")

# Save A_static directly (skip Neo4j for .npy generation)
# This is the EXACT same adjacency PriSTI uses
A_static = adj.copy()
np.save(f"{OUTPUT_DIR}/A_static.npy", A_static)
print(f"✅ A_static.npy: {A_static.shape}, {np.count_nonzero(A_static)} non-zero")

# Save var_mean and var_std
var_mean = np.array([n['temporal_var_mean'] for n in nodes_metadata])
var_std = np.array([n['temporal_var_std'] for n in nodes_metadata])
np.save(f"{OUTPUT_DIR}/var_mean.npy", var_mean)
np.save(f"{OUTPUT_DIR}/var_std.npy", var_std)
print(f"✅ var_mean.npy: {var_mean.shape}")
print(f"✅ var_std.npy: {var_std.shape}")

# Save distance matrix (copy for the model)
np.save(f"{OUTPUT_DIR}/metr_la_dist.npy", dist_matrix)
print(f"✅ metr_la_dist.npy: {dist_matrix.shape}")



✅ nodes_metadata.json: 207 nodes
✅ relations_metadata.json: 776 relations
✅ A_static.npy: (207, 207), 1515 non-zero
✅ var_mean.npy: (207,)
✅ var_std.npy: (207,)
✅ metr_la_dist.npy: (207, 207)


In [8]:
# ============================================================
# CELL 8: Validation
# ============================================================
print("\n" + "=" * 60)
print("VALIDATION")
print("=" * 60)

# Verify A_static matches what PriSTI would compute
print(f"\nA_static:")
print(f"  Shape: {A_static.shape}")
print(f"  Symmetric: {np.allclose(A_static, A_static.T)}")
print(f"  Diagonal = 0: {np.all(np.diag(A_static) == 0)}")
print(f"  Non-zero: {np.count_nonzero(A_static)}")
print(f"  Value range: [{A_static[A_static>0].min():.4f}, {A_static.max():.4f}]")

print(f"\nAnomaly thresholds (first 5 sensors):")
for i in range(5):
    thr = var_mean[i] + 3 * var_std[i]
    print(f"  Sensor {i} ({sensor_ids[i]}): μ={var_mean[i]:.1f}, σ={var_std[i]:.1f}, threshold={thr:.1f}")

print(f"\n{'='*60}")
print(f"FILES SAVED TO: {OUTPUT_DIR}")
print(f"{'='*60}")




VALIDATION

A_static:
  Shape: (207, 207)
  Symmetric: False
  Diagonal = 0: True
  Non-zero: 1515
  Value range: [0.1001, 0.9998]

Anomaly thresholds (first 5 sensors):
  Sensor 0 (773869): μ=51.2, σ=134.4, threshold=454.6
  Sensor 1 (767541): μ=6.2, σ=15.4, threshold=52.3
  Sensor 2 (767542): μ=39.2, σ=107.8, threshold=362.5
  Sensor 3 (717447): μ=30.3, σ=58.5, threshold=205.9
  Sensor 4 (717446): μ=41.5, σ=53.4, threshold=201.7

FILES SAVED TO: /kaggle/working/metrla_metadata


In [9]:
# ============================================================
# CELL 9: Download
# ============================================================
import shutil
shutil.make_archive('/kaggle/working/metrla_metadata_all', 'zip', OUTPUT_DIR)
print(f"\n📦 Download: /kaggle/working/metrla_metadata_all.zip")
print(f"   Contains: nodes_metadata.json, relations_metadata.json,")
print(f"             A_static.npy, var_mean.npy, var_std.npy, metr_la_dist.npy")
print(f"\n🎉 METR-LA metadata generation complete!")


📦 Download: /kaggle/working/metrla_metadata_all.zip
   Contains: nodes_metadata.json, relations_metadata.json,
             A_static.npy, var_mean.npy, var_std.npy, metr_la_dist.npy

🎉 METR-LA metadata generation complete!
